# Q-Orbit Phase 1: Classical Baseline Portfolio Optimization

This notebook demonstrates the classical Markowitz Mean-Variance optimization that serves as our baseline for comparison with quantum-enhanced methods.

## Objectives
1. Fetch historical price data
2. Calculate returns and basic statistics
3. Optimize portfolio using Maximum Sharpe Ratio
4. Visualize results
5. Establish baseline performance metrics

In [ ]:
# Setup
import sys
sys.path.append('../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils.data_loader import DataLoader, get_sample_portfolio
from classical.baseline import MarkowitzOptimizer
from utils.visualization import PortfolioVisualizer

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Collection

Let's start by fetching historical price data for a diversified portfolio of stocks.

In [ ]:
# Initialize data loader
loader = DataLoader()

# Define portfolio
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 
           'JPM', 'JNJ', 'V', 'PG', 'NVDA']

start_date = '2020-01-01'
end_date = '2024-01-01'

print(f"Fetching data for {len(tickers)} assets...")
print(f"Period: {start_date} to {end_date}")
print(f"Tickers: {', '.join(tickers)}")

In [ ]:
# Fetch price data
prices = loader.fetch_price_data(tickers, start_date, end_date)

print(f"\nData shape: {prices.shape}")
print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
print(f"\nFirst 5 rows:")
prices.head()

In [ ]:
# Calculate returns
returns = loader.calculate_returns(prices)

print(f"Returns shape: {returns.shape}")
print(f"\nReturns statistics:")
returns.describe()

## 2. Exploratory Data Analysis

In [ ]:
# Plot price evolution
normalized_prices = prices / prices.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 7))
normalized_prices.plot(ax=ax, linewidth=2, alpha=0.7)
ax.set_title('Normalized Price Evolution (Base = 100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price')
ax.legend(loc='best', ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Risk-return scatter
viz = PortfolioVisualizer()
viz.plot_risk_return_scatter(returns)

In [ ]:
# Correlation matrix
viz.plot_correlation_matrix(returns)

## 3. Portfolio Optimization

Now let's optimize the portfolio using different strategies:
1. **Maximum Sharpe Ratio**: Best risk-adjusted returns
2. **Minimum Variance**: Lowest risk portfolio
3. **Efficient Frontier**: Trade-off between risk and return

In [ ]:
# Initialize optimizer
optimizer = MarkowitzOptimizer(risk_free_rate=0.04)

### 3.1 Maximum Sharpe Ratio Portfolio

In [ ]:
# Optimize for max Sharpe ratio
weights_sharpe = optimizer.optimize_max_sharpe(returns)

print("="*60)
print("MAXIMUM SHARPE RATIO PORTFOLIO")
print("="*60)

print("\nOptimal Weights:")
for ticker, weight in optimizer.weights.items():
    if weight > 0.01:  # Only show significant allocations
        print(f"  {ticker:6s}: {weight:6.2%}")

print("\nPerformance Metrics:")
for metric, value in optimizer.get_performance_summary().items():
    print(f"  {metric:20s}: {value}")

In [ ]:
# Visualize weights
viz.plot_weights(optimizer.weights, title="Max Sharpe Ratio Portfolio Weights")

### 3.2 Minimum Variance Portfolio

In [ ]:
# Optimize for minimum variance
weights_minvar = optimizer.optimize_min_variance(returns)

print("="*60)
print("MINIMUM VARIANCE PORTFOLIO")
print("="*60)

print("\nOptimal Weights:")
for ticker, weight in optimizer.weights.items():
    if weight > 0.01:
        print(f"  {ticker:6s}: {weight:6.2%}")

print("\nPerformance Metrics:")
for metric, value in optimizer.get_performance_summary().items():
    print(f"  {metric:20s}: {value}")

### 3.3 Efficient Frontier

In [ ]:
# Generate efficient frontier
print("Generating efficient frontier (this may take a minute)...")
risks, frontier_returns, weights_list = optimizer.generate_efficient_frontier(returns, n_points=30)

# Re-optimize for max Sharpe to get the optimal point
optimizer.optimize_max_sharpe(returns)

# Plot
viz.plot_efficient_frontier(
    risks,
    frontier_returns,
    weights_list,
    highlight_portfolio={
        'risk': optimizer.performance['annual_volatility'],
        'return': optimizer.performance['annual_return']
    }
)

## 4. Performance Analysis

In [ ]:
# Load benchmark (S&P 500)
benchmark_prices = loader.get_benchmark_data(start_date, end_date, 'SPY')
benchmark_returns = loader.calculate_returns(benchmark_prices)['SPY']

print(f"Benchmark (S&P 500) loaded: {len(benchmark_returns)} data points")

In [ ]:
# Plot cumulative returns vs benchmark
viz.plot_cumulative_returns(
    returns,
    optimizer.weights,
    benchmark_returns=benchmark_returns
)

In [ ]:
# Comprehensive dashboard
viz.create_performance_dashboard(
    returns,
    optimizer.weights,
    optimizer.get_performance_summary()
)

## 5. Baseline Metrics Summary

Let's save these baseline metrics for comparison with quantum-enhanced results.

In [ ]:
# Calculate benchmark metrics
benchmark_annual_return = benchmark_returns.mean() * 252
benchmark_annual_vol = benchmark_returns.std() * np.sqrt(252)
benchmark_sharpe = (benchmark_annual_return - 0.04) / benchmark_annual_vol

# Create comparison table
comparison = pd.DataFrame({
    'Portfolio (Max Sharpe)': [
        optimizer.performance['annual_return'],
        optimizer.performance['annual_volatility'],
        optimizer.performance['sharpe_ratio'],
        optimizer.performance['max_drawdown']
    ],
    'Benchmark (S&P 500)': [
        benchmark_annual_return,
        benchmark_annual_vol,
        benchmark_sharpe,
        np.nan  # Not calculated for benchmark
    ]
}, index=['Annual Return', 'Annual Volatility', 'Sharpe Ratio', 'Max Drawdown'])

print("\n" + "="*70)
print("BASELINE PERFORMANCE COMPARISON")
print("="*70)
print(comparison.to_string())

# Save to CSV
comparison.to_csv('../data/baseline_metrics.csv')
print("\n✓ Baseline metrics saved to data/baseline_metrics.csv")

## 6. Next Steps

Now that we have established a classical baseline, the next phases are:

1. **Phase 2: LLM Sentiment Analysis**
   - Fetch financial news
   - Apply FinBERT sentiment scoring
   - Convert sentiment to portfolio constraints

2. **Phase 3: Quantum QAOA**
   - Learn QAOA fundamentals
   - Map portfolio problem to Ising Hamiltonian
   - Implement quantum optimization

3. **Phase 4: Hybrid Integration**
   - Combine sentiment constraints with QAOA
   - Benchmark quantum vs classical
   - Evaluate "Quantum Advantage"

---

### 📝 Key Takeaways

- Classical optimization provides strong baseline
- Sharpe ratio typically 1.0-2.0 for diversified portfolios
- Correlation analysis reveals diversification opportunities
- Efficient frontier shows risk-return trade-offs

**This baseline will be used to measure potential quantum advantage in later phases!**